# planttype leaf-gas-exchange loop: does it need relaxation? (debug/tl-convergence-isolation)

**Research/debug notebook -- lives on branch `debug/tl-convergence-isolation`, not intended to be merged.**

This is the third and last of the three non-convergence sources identified from
`Examples/logs/ran_22_k7.log`'s DEBUG messages:

| module | log occurrences | status |
|---|---|---|
| `mlm_canopy.run` (outer Picard loop) | 627 | fixed (`gam_floor` 0.25 -> 0.01) |
| `interception.run` (wet-leaf loop) | 183 | fixed (gamma relaxation added, see `debug_interception_relaxation.ipynb`) |
| `planttype.leaf_gas_exchange` (sunlit/shaded dry-leaf loop) | 337 | **this notebook** |

`PlantType.leaf_gas_exchange()` (`pyAPES/planttype/planttype.py`, the
`while err > 0.01 and iter_no < itermax:` block around lines 502-545) solves sunlit and
shaded leaf temperature by the same kind of plain fixed-point iteration interception used to
have -- **no relaxation**. This notebook copies that loop into a standalone sandbox function,
adds a `gamma`/`gamma_floor` relaxation knob (same oscillation-adaptive design validated for
interception: start undamped at `gamma=1.0`, only back off once the error actually grows), and
sweeps it across the forcing already captured in
`debug_captures/forcing_exploration/planttype_forcing_samples.pkl` (1392 timesteps, June 2022,
captured at `iter_no==2`). **No new capture run is needed.**

There are 3 planttypes in the FI-Ran clear-cut parameterisation (`spruce`, `decid`, `shrubs`)
and 2 leaf types (`sunlit`, `shaded`) per planttype -- this notebook tests **all of them
separately** (not just one representative species), since `photop`/`leafp` differ by species
and could converge differently.

This notebook does **not** modify `pyAPES/planttype/planttype.py`. If relaxation reliably
helps, port the equivalent minimal change into the real source afterwards, then re-verify.

**Caveat on the initial guess for `Tl_sl`/`Tl_sh`:** these attributes aren't set in
`PlantType.__init__()` at all -- they only exist after `PlantType.run()` has executed once
with the energy balance on. In the real model, `CanopyModel._restore()` resets them to air
temperature (`pt.Tl_sh = T.copy(); pt.Tl_sl = T.copy()`) once at the start of each timestep,
same cold-start pattern as `interception.Tl_wet`. This notebook does the same: every sample
starts from `Tl_ini = air_temperature`.

In [ ]:
import os
import sys

assert os.path.basename(os.getcwd()) == 'debug', (
    f"expected to run with cwd=debug/ (this notebook's own directory), got {os.getcwd()!r} -- adjust paths below if not")
sys.path.insert(0, os.path.abspath('..'))

import pickle
import copy
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt

FORCING_CAPTURE_DIR = '../Examples/debug_captures/forcing_exploration'
CAPTURE_FILE = os.path.join(FORCING_CAPTURE_DIR, 'planttype_forcing_samples.pkl')

with open(CAPTURE_FILE, 'rb') as f:
    records = pickle.load(f)

print(f'{len(records)} captured planttype-forcing records loaded from {CAPTURE_FILE}')
outcome_counts = pd.Series([r['outcome'] for r in records]).value_counts()
print('outer mlm_canopy outcome breakdown:')
print(outcome_counts)

In [ ]:
# self.leafp['lt'], self.Photo_model, self.photop, self.lad are static/structural PlantType
# attributes (not part of per-timestep forcing, so not captured). Build the real CanopyModel
# to get the 3 real PlantType instances -- cheap, object construction only.
from pyAPES.soil.soil import Soil_1D
from pyAPES.canopy.mlm_canopy import CanopyModel
from pyAPES.parameters.mlm_parameters_FI_Ran import gpara, cpara, spara

soil = Soil_1D(spara)
canopy_model = CanopyModel(cpara, dz_soil=soil.grid['dz'])

# CanopyModel builds planttypes sorted by name (see mlm_canopy.py CanopyModel.__init__),
# so zip with the same sorted key order to get a name -> PlantType mapping
pt_names = sorted(cpara['planttypes'].keys())
planttypes = dict(zip(pt_names, canopy_model.planttypes))

print('planttypes:', list(planttypes.keys()))
for name, pt in planttypes.items():
    print(f'  {name}: lad nonzero layers = {int(np.sum(pt.lad > 0))}, LAImax = {pt.LAImax}')

## EXPERIMENTAL sandbox -- do not treat as source of truth

Copy of the energy-balance loop from `PlantType.leaf_gas_exchange()`
(`pyAPES/planttype/planttype.py`, roughly lines 498-545), with the same relaxation design
already validated for interception: start undamped (`gamma=1.0`), and only apply damping
(halving `gamma` down to `gamma_floor`) once the error actually grows after a short warm-up
(`osc_check_after` iterations) -- this avoids slowing down the ~99% of cases that already
converge fine unrelaxed.

**This copy will drift from the source over time.** Any fix validated here must be ported back
into `planttype.py` as a real patch, then re-verified.

In [ ]:
from pyAPES.leaf.boundarylayer import leaf_boundary_layer_conductance
from pyAPES.microclimate.micromet import e_sat, latent_heat
from pyAPES.leaf.photo import set_photo_forcing
from pyAPES.utils.constants import MOLAR_MASS_H2O, SPECIFIC_HEAT_AIR, EPS, PAR_TO_UMOL, H2O_CO2_RATIO


def solve_leaf_temperature(pt, record, leaftype, gamma=1.0, gamma_floor=None,
                            osc_check_after=None, max_iter=20, max_err=0.01):
    # Sandbox copy of PlantType.leaf_gas_exchange()'s energy-balance loop. See markdown above.
    T = np.array(record['air_temperature'], ndmin=1)
    H2O = np.array(record['h2o'], ndmin=1)
    P = record['air_pressure']
    U = record['wind_speed']
    CO2 = record['co2']

    Qp = record['par'][leaftype]['incident'] * PAR_TO_UMOL
    SWabs = record['par'][leaftype]['absorbed'] + record['nir'][leaftype]['absorbed']
    LWnet = record['lw']['net_leaf']
    Rabs = SWabs + LWnet
    gr = record['lw']['radiative_conductance']
    Tl_ave = record['average_leaf_temperature']

    # cold-start initial guess, matching CanopyModel._restore() (see markdown caveat)
    Tl_ini = T.copy()
    ic = np.where(np.abs(LWnet) > 0.0)

    Tl = Tl_ini.copy()
    Told = Tl.copy()

    esat, s = e_sat(Tl)
    s = s / P
    Dleaf = esat / P - H2O
    Lv = latent_heat(T) * MOLAR_MASS_H2O

    gam = gamma
    traj = []
    err = 999.0
    iter_no = 0
    while err > max_err and iter_no < max_iter:
        iter_no += 1
        Told = Tl.copy()

        gb_h, gb_c, gb_v = leaf_boundary_layer_conductance(
            U, pt.leafp['lt'], T, 0.5 * (Tl + Told) - T, P)

        pt.photo_forcing = set_photo_forcing(pt.photo_forcing, Qp, Tl, Dleaf, CO2, gb_c, gb_v, P)
        photo_results = pt.Photo_model.run(pt.photo_forcing, pt.photop)

        gsv = H2O_CO2_RATIO * photo_results['gs_opt']
        geff_v = np.where(Dleaf > 0.0, (gb_v * gsv) / (gb_v + gsv), gb_v)

        Tl_candidate = Tl.copy()
        Tl_candidate[ic] = (
            Rabs[ic] + SPECIFIC_HEAT_AIR * gr[ic] * Tl_ave[ic] + SPECIFIC_HEAT_AIR * gb_h[ic] * T[ic]
            - Lv[ic] * geff_v[ic] * Dleaf[ic] + Lv[ic] * s[ic] * geff_v[ic] * Told[ic]
        ) / (SPECIFIC_HEAT_AIR * (gr[ic] + gb_h[ic]) + Lv[ic] * s[ic] * geff_v[ic])

        Tl = Told.copy()
        Tl[ic] = Told[ic] + gam * (Tl_candidate[ic] - Told[ic])
        err = np.nanmax(np.abs(Tl - Told))

        if gamma_floor is not None and osc_check_after is not None and iter_no > osc_check_after:
            prev_err = traj[-1]['err'] if traj else None
            if prev_err is not None and err > prev_err:
                Tl[ic] = 0.5 * (Told[ic] + Tl[ic])
                gam = max(gam / 2, gamma_floor)
                err = np.nanmax(np.abs(Tl - Told))

        esat, s = e_sat(Tl)
        s = s / P
        Dleaf = esat / P - H2O

        traj.append({'iter': iter_no, 'err': err, 'gam': gam, 'Tl_mean': float(np.nanmean(Tl[ic]))})

    converged = err <= max_err and iter_no < max_iter
    return {'traj': traj, 'iterNo': iter_no, 'err': err, 'converged': converged, 'Tl': Tl}

In [ ]:
def sweep_planttype(records, planttypes, leaftypes=('sunlit', 'shaded'), **kwargs):
    rows = []
    for r in records:
        for pt_name, pt in planttypes.items():
            for leaftype in leaftypes:
                res = solve_leaf_temperature(pt, r, leaftype, **kwargs)
                rows.append({
                    'timestamp': r['timestamp'],
                    'hour': r['timestamp'].hour,
                    'outer_outcome': r['outcome'],
                    'planttype': pt_name,
                    'leaftype': leaftype,
                    'iterNo': res['iterNo'],
                    'final_err': res['err'],
                    'converged': res['converged'],
                })
    return pd.DataFrame(rows)


baseline_df = sweep_planttype(records, planttypes, gamma=1.0, gamma_floor=None, max_iter=20, max_err=0.01)

n_fail = int((~baseline_df['converged']).sum())
print(f"baseline (gamma=1.0, i.e. unmodified source): {n_fail}/{len(baseline_df)} "
      f"planttype leaf-temperature solves fail to converge within 20 iterations")
print()
print('non-convergence count by planttype x leaftype:')
print(baseline_df.loc[~baseline_df['converged']].groupby(['planttype', 'leaftype']).size())
baseline_df.sort_values('final_err', ascending=False).head(10)

In [ ]:
COLOR_CONVERGED = '#0ca30c'
COLOR_NONCONVERGED = '#d03b3b'

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, pt_name in zip(axes, planttypes.keys()):
    sub = baseline_df[baseline_df['planttype'] == pt_name].sort_values('final_err', ascending=False).reset_index(drop=True)
    colors = [COLOR_CONVERGED if c else COLOR_NONCONVERGED for c in sub['converged']]
    ax.bar(range(len(sub)), sub['final_err'], color=colors, width=1.0)
    ax.axhline(0.01, color='k', ls='--', lw=0.8)
    ax.set_yscale('log')
    ax.set_title(f"{pt_name}: {int(sub['converged'].sum())}/{len(sub)} converged")
    ax.set_xlabel('sample (sorted, sunlit+shaded combined)')
axes[0].set_ylabel('final |Tl - Told| (log scale)')
plt.tight_layout()
plt.show()

## Try adding relaxation

Edit `GAMMA_FLOOR` / `OSC_CHECK_AFTER` below (oscillation-adaptive damping, same design as
`interception.py`'s fix: undamped by default, only backs off once the error grows) and re-run
to try different values against all captured timesteps x planttypes x leaftypes at once.

In [ ]:
# --- user-adjustable knobs ---
GAMMA = 1.0          # starting relaxation factor (1.0 = undamped, matches unmodified source)
GAMMA_FLOOR = 0.25    # set to None to disable oscillation-adaptive damping entirely
OSC_CHECK_AFTER = 5
MAX_ITER = 20
MAX_ERR = 0.01

relaxed_df = sweep_planttype(records, planttypes, gamma=GAMMA, gamma_floor=GAMMA_FLOOR,
                              osc_check_after=OSC_CHECK_AFTER, max_iter=MAX_ITER, max_err=MAX_ERR)

n_fail_relaxed = int((~relaxed_df['converged']).sum())
print(f"relaxed (gamma={GAMMA}, gamma_floor={GAMMA_FLOOR}): "
      f"{n_fail_relaxed}/{len(relaxed_df)} fail to converge within {MAX_ITER} iterations")
print()
print('non-convergence count by planttype x leaftype:')
print(relaxed_df.loc[~relaxed_df['converged']].groupby(['planttype', 'leaftype']).size() if n_fail_relaxed else 'none')
relaxed_df.sort_values('final_err', ascending=False).head(10)

In [ ]:
summary = pd.DataFrame({
    'baseline (gamma=1.0)': [n_fail, len(baseline_df) - n_fail,
                              baseline_df['iterNo'].mean(), baseline_df['iterNo'].median()],
    f'relaxed (floor={GAMMA_FLOOR})': [n_fail_relaxed, len(relaxed_df) - n_fail_relaxed,
                              relaxed_df['iterNo'].mean(), relaxed_df['iterNo'].median()],
}, index=['n_non_converged', 'n_converged', 'mean_iterations', 'median_iterations'])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(baseline_df['iterNo'], bins=range(1, MAX_ITER + 2), alpha=0.6, label='baseline (gamma=1.0)', color=COLOR_NONCONVERGED)
ax.hist(relaxed_df['iterNo'], bins=range(1, MAX_ITER + 2), alpha=0.6, label=f'relaxed (floor={GAMMA_FLOOR})', color=COLOR_CONVERGED)
ax.set_xlabel('iterations to converge (or max_iter if not converged)')
ax.set_ylabel('count of samples')
ax.legend()
ax.set_title('iterations-to-converge distribution')

worst_idx = baseline_df['final_err'].idxmax()
worst_row = baseline_df.loc[worst_idx]
worst_record = next(r for r in records if r['timestamp'] == worst_row['timestamp'])
worst_pt = planttypes[worst_row['planttype']]
worst_leaftype = worst_row['leaftype']

res_baseline = solve_leaf_temperature(worst_pt, worst_record, worst_leaftype, gamma=1.0, max_iter=MAX_ITER, max_err=MAX_ERR)
res_relaxed = solve_leaf_temperature(worst_pt, worst_record, worst_leaftype, gamma=GAMMA, gamma_floor=GAMMA_FLOOR,
                                      osc_check_after=OSC_CHECK_AFTER, max_iter=MAX_ITER, max_err=MAX_ERR)

ax = axes[1]
ax.plot(range(1, len(res_baseline['traj']) + 1), [s['err'] for s in res_baseline['traj']],
        marker='o', ms=3, color=COLOR_NONCONVERGED, label='baseline (gamma=1.0)')
ax.plot(range(1, len(res_relaxed['traj']) + 1), [s['err'] for s in res_relaxed['traj']],
        marker='o', ms=3, color=COLOR_CONVERGED, label=f'relaxed (floor={GAMMA_FLOOR})')
ax.axhline(MAX_ERR, color='k', ls='--', lw=0.8, label='max_err')
ax.set_yscale('log')
ax.set_xlabel('iteration')
ax.set_ylabel('|Tl - Told|')
ax.set_title(f"worst baseline sample: {worst_row['planttype']}/{worst_row['leaftype']} {worst_row['timestamp']}")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Next steps

If `GAMMA_FLOOR`/`OSC_CHECK_AFTER` above reliably reduces non-convergence and
iterations-to-converge across all 3 planttypes without materially raising mean iterations for
the already-converging majority, port the equivalent minimal change into the real
`while err > 0.01 and iter_no < itermax:` loop in `PlantType.leaf_gas_exchange()`
(`pyAPES/planttype/planttype.py`, lines ~502-545) -- not this notebook, following the same
pattern already applied to `pyAPES/canopy/interception.py`. Then re-run
`Examples/capture_forcing_exploration.py` and confirm the `pyAPES.planttype.planttype
leaf_gas_exchange` "Maximum number of iterations reached" count in
`logs/capture_forcing_exploration.log` drops from its current baseline.